In [2]:
import torch
import gc

if "model" in locals(): del model
if "trainer" in locals(): del trainer
if "tokenizer" in locals(): del tokenizer

gc.collect()
torch.cuda.empty_cache()

free_memory = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Free VRAM: {free_memory:.2f} GB. (Need ~5.5 GB)")

Free VRAM: 14.46 GB. (Need ~5.5 GB)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

!pip install unsloth_zoo
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install xformers trl peft accelerate bitsandbytes

In [8]:
import torch
from unsloth import FastLanguageModel


save_path = "/content/drive/MyDrive/MedicalFineTune"
max_seq_length = 2048
load_in_4bit = True


model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


FastLanguageModel.for_inference(model)


alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


print("\n" + "="*50)
print("CHAT DOCTOR ACTIVE - Type 'quit' to exit")
print("="*50)

while True:
    user_query = input("\nAsk Chat Doctor: ")

    if user_query.lower() in ['quit', 'exit', 'stop']:
        print("Closing session. Stay healthy!")
        break


    formatted_prompt = alpaca_prompt.format(user_query, "", "")

    inputs = tokenizer(
        [formatted_prompt],
        return_tensors = "pt"
    ).to("cuda")


    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        use_cache = True,
        temperature = 0.5,
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.eos_token_id,
    )


    decoded_output = tokenizer.batch_decode(outputs)[0]

    try:

        response_only = decoded_output.split("### Response:")[1].replace(tokenizer.eos_token, "").strip()
        print(f"\n[Chat Doctor]: {response_only}\n")
    except IndexError:
        print(f"\n[Chat Doctor]: {decoded_output}\n")

    print("-" * 50)

Unsloth: Already have LoRA adapters! We shall skip this step.



CHAT DOCTOR ACTIVE - Type 'quit' to exit

Ask Chat Doctor: what is pneumonia

[Chat Doctor]: Pneumonia is a lung infection that can be caused by bacteria, viruses, or fungi. It is characterized by inflammation of the air sacs (alveoli) in the lungs, which can lead to difficulty breathing, chest pain, and other symptoms. Pneumonia can be treated with antibiotics, antiviral medications, or antifungal drugs, depending on the cause of the infection. It is important to seek medical attention if you suspect you have pneumonia, as it can be a serious condition if left untreated.

--------------------------------------------------

Ask Chat Doctor: how to treat fever

[Chat Doctor]: Take fever medicine

--------------------------------------------------

Ask Chat Doctor: how to treat diabetes

[Chat Doctor]: The first step is to understand what diabetes is. Diabetes is a condition where the body doesn't produce or respond to insulin. Insulin is a hormone that helps the body use glucose, a typ